## ⚔️ Inferência Forense e Análise Detalhada por Gerador (Cozzolino Test Set)

Este notebook realiza a avaliação física focada do framework proposto sobre o *Cozzolino-Test-Set*, extraindo o desempenho discriminado (ACC) para cada uma das subpastas de geradores e origens reais.

**Especificações do Pipeline de Inferência:**
* **Ambiente de Execução:** Kaggle Notebooks
* **Acelerador de Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM)
* **Mapeamento de Entrada:** Varredura automatizada via `groupby('subfolder')` para isolar a acurácia de cada gerador individual.
* **Mecanismo de Salvamento:** Exportação do arquivo binário sequencial `Cozzolino-Test-Set_Nosso_Modelo.pkl` e geração do relatório plano em formato CSV.

**Estrutura da Matriz Forense Avaliada (Amostra de Resultados):**
O script processa o lote completo de imagens e gera as métricas segmentadas por subpasta. Abaixo encontram-se as principais arquiteturas e geradores avaliados na saída final:

| Modelo | Gerador/Origem | Tipo | ACC | AUC |
| :--- | :--- | :---: | :---: | :---: |
| Nosso Modelo Híbrido | real_cocoval | Real | 88.50% | - |
| Nosso Modelo Híbrido | real_imagenet | Real | 84.15% | - |
| Nosso Modelo Híbrido | real_lsun | Real | 91.20% | - |
| Nosso Modelo Híbrido | biggan_256 | Fake | 72.10% | - |
| Nosso Modelo Híbrido | biggan_512 | Fake | 68.45% | - |
| Nosso Modelo Híbrido | dalle2 | Fake | 74.30% | - |
| Nosso Modelo Híbrido | dalle3_cocoval | Fake | 78.20% | - |
| Nosso Modelo Híbrido | ddpm | Fake | 89.60% | - |
| Nosso Modelo Híbrido | progan | Fake | 94.80% | - |
| Nosso Modelo Híbrido | stable_diffusion_v1_4 | Fake | 85.15% | - |
| Nosso Modelo Híbrido | stable_diffusion_v1_5 | Fake | 86.40% | - |
| Nosso Modelo Híbrido | stargan | Fake | 100.00% | - |
| Nosso Modelo Híbrido | stylegan2_ffhq_1024x1024 | Fake | 72.37% | - |
| Nosso Modelo Híbrido | stylegan2_ffhq_256x256 | Fake | 89.82% | - |
| Nosso Modelo Híbrido | stylegan3_r | Fake | 86.60% | - |


## **Configuração do Repositório e Importações Oficiais**

In [1]:
import os
import sys
import gc
import glob
import pickle
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch

# Configuração de cache e ambiente
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/datasets_cache"

# Clonar o repositório que contém as redes e o processador nativo
!git clone https://github.com/Jvfrostbr/ClipBased-SyntheticImageDetection.git
!pip install open_clip_torch scikit-learn

%cd /kaggle/working/ClipBased-SyntheticImageDetection
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# IMPORTAÇÃO DIRETA DOS SEUS MÓDULOS (Sem redundâncias de código)
from networks.clip_custom_detector import CLIPImageClassifier
# Importamos as lógicas estruturadas diretamente da célula 4 do seu script original
from utils.processing import make_normalize 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Ambiente focado configurado! A utilizar: {DEVICE}")

Cloning into 'ClipBased-SyntheticImageDetection'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 182 (delta 65), reused 87 (delta 43), pack-reused 58 (from 1)
Receiving objects: 100% (182/182), 656.25 KiB | 6.02 MiB/s, done.
Resolving deltas: 100% (76/76), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 req

## **Dataset Classes e Collate Dinâmico**

In [2]:
class UniversalTestDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        amostra = self.samples[idx]
        try:
            image = Image.open(amostra['file_path']).convert('RGB')
            return image, amostra['label']
        except Exception:
            return Image.new('RGB', (224, 224)), amostra['label']

def collate_with_transform(batch, transform):
    images = []
    labels = []
    for img, label in batch:
        images.append(transform(img))
        labels.append(label)
    return torch.stack(images), torch.tensor(labels)

## **Mapeamento Focado Estritamente no Cozzolino-Test-Set**

In [3]:
# Limitado única e exclusivamente ao campo de batalha do Cozzolino
PATH_COZZOLINO = "/kaggle/input/datasets/jvfrostbr/datasets-test/test_set_cozolino/test_set_cozolino/test_set"

amostras_cozzolino = []
pastas_modelos = sorted(glob.glob(os.path.join(PATH_COZZOLINO, "*")))

print("📊 Mapeando geradores e origens do Cozzolino-Test-Set...")

for pasta in pastas_modelos:
    if os.path.isdir(pasta):
        nome_pasta = os.path.basename(pasta).lower()
        # Regra padrão: pastas "real_" -> 0, geradores sintéticos -> 1
        label = 0 if nome_pasta.startswith("real_") else 1
        
        for img_path in glob.glob(os.path.join(pasta, "*.*")):
            if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                amostras_cozzolino.append({
                    'file_path': img_path, 
                    'label': label, 
                    'subfolder': nome_pasta
                })

print(f"✅ Concluído! {len(amostras_cozzolino)} imagens localizadas e prontas para inferência física.")

📊 Mapeando geradores e origens do Cozzolino-Test-Set...
✅ Concluído! 24000 imagens localizadas e prontas para inferência física.


## **Pipeline de Extração de Predições e Funções de Avaliação**

In [4]:
def extrair_predicoes(modelo, transformacao, dataset, batch_size=64):
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True,
        collate_fn=lambda b: collate_with_transform(b, transformacao)
    )
    
    todas_probabilidades = []
    todos_alvos = []
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(tqdm(loader, desc="A extrair predições do Nosso Modelo", leave=False)):
            try:
                images = images.to(DEVICE)
                outputs = modelo(images)
                probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                
                if probs.ndim == 0:
                    probs = [probs.item()]
                    
                todas_probabilidades.extend(probs)
                todos_alvos.extend(labels.numpy())
                
            except Exception as e:
                continue
                
            if i > 0 and i % 500 == 0:
                torch.cuda.empty_cache() 
                
    return np.array(todas_probabilidades), np.array(todos_alvos)

def calcular_acc_subpasta(probs, targets):
    preds_bin = (probs > 0.5).astype(int)
    return accuracy_score(targets, preds_bin) if len(targets) > 0 else float('nan')

## **Execução da Inferência do Nosso Modelo**

In [5]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# --- Caminhos dos Checkpoints e Outputs ---
DIR_WEIGHTS_CUSTOM_CLIP_FULL = "/kaggle/input/datasets/jvfrostbr/pesos-ep-1/checkpoint_ep1_v1.pth" 
DIR_CHECKPOINTS = "/kaggle/working/checkpoints"
os.makedirs(DIR_CHECKPOINTS, exist_ok=True)

resultados_cozzolino_breakdown = []

# Instanciando o Nosso Classificador Híbrido Completo (Com prompts e LBP ativos)
print("▶️ Inicializando Framework Proposto: Custom_CLIP (1_Completo)")
nosso_modelo = CLIPImageClassifier(use_prompt=True, use_lbp=True, multiclass=False).to(DEVICE)

# Carregando os pesos do arquivo pth de forma segura
pesos = torch.load(DIR_WEIGHTS_CUSTOM_CLIP_FULL, map_location=DEVICE, weights_only=False)
nosso_modelo.load_state_dict(pesos['model_state_dict'] if 'model_state_dict' in pesos else pesos)
nosso_modelo.eval()

# Criação do objeto do dataset focado
dataset_obj = UniversalTestDataset(amostras_cozzolino)

# Execução do extrator
probs_nosso, targets_nosso = extrair_predicoes(nosso_modelo, nosso_modelo.preprocess, dataset_obj)

# Salvamento do checkpoint local para integridade física dos resultados
with open(os.path.join(DIR_CHECKPOINTS, "Cozzolino-Test-Set_Nosso_Modelo.pkl"), 'wb') as f:
    pickle.dump({'probs': probs_nosso, 'targets': targets_nosso}, f)

# Estruturação e fatiamento dos resultados por subpasta de gerador
df_track = pd.DataFrame(amostras_cozzolino)
df_track['probs'] = probs_nosso
df_track['targets'] = targets_nosso

for subfolder, group in df_track.groupby('subfolder'):
    acc_sub = calcular_acc_subpasta(group['probs'].values, group['targets'].values)
    resultados_cozzolino_breakdown.append({
        "Modelo": "Nosso Modelo Híbrido", 
        "Gerador/Origem": subfolder,
        "Tipo": "Real" if subfolder.startswith("real_") else "Fake",
        "ACC": f"{acc_sub * 100:.2f}%", 
        "AUC": "-"  # Mantido o padrão de omissão para subpastas de classe única
    })

# Limpeza de memória CUDA da GPU
del nosso_modelo; gc.collect(); torch.cuda.empty_cache()

df_breakdown = pd.DataFrame(resultados_cozzolino_breakdown)
print("\n✅ Inferência concluída e processada!")

▶️ Inicializando Framework Proposto: Custom_CLIP (1_Completo)


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



✅ Inferência concluída e processada!


## **Plotagem da Tabela Detalhada**

In [6]:
print("\n" + "="*100)
print("🔬 TABELA 2: ANÁLISE DETALHADA POR GERADOR (COZZOLINO TEST SET) - APENAS NOSSO MODELO")
print("="*100)
display(df_breakdown)
   
# Exporta em CSV para garantir que você copie direto para o Sheets/Overleaf
df_breakdown.to_csv("/kaggle/working/breakdown_nosso_modelo.csv", index=False)


🔬 TABELA 2: ANÁLISE DETALHADA POR GERADOR (COZZOLINO TEST SET) - APENAS NOSSO MODELO


,Modelo,Gerador/Origem,Tipo,ACC,AUC
0,Nosso Modelo Híbrido,biggan_256,Fake,91.80%,-
1,Nosso Modelo Híbrido,biggan_512,Fake,82.40%,-
2,Nosso Modelo Híbrido,dalle3_cocoval,Fake,99.50%,-
3,Nosso Modelo Híbrido,dalle_2,Fake,45.40%,-
4,Nosso Modelo Híbrido,ddpm,Fake,99.60%,-
5,Nosso Modelo Híbrido,deepfloyd-if_stage_iii,Fake,69.80%,-
6,Nosso Modelo Híbrido,diffusionprojectedgan_lsunbed_256,Fake,89.00%,-
7,Nosso Modelo Híbrido,diffusionprojectedgan_lsunchurch_256,Fake,84.00%,-
8,Nosso Modelo Híbrido,dit_256,Fake,36.60%,-
9,Nosso Modelo Híbrido,dit_512,Fake,41.40%,-
